## Inverse Kinematics (algebraic, symbolic) - Lab 1 R-R-P robot
Given the tip position $(x, y, z)$, find $(q_1, q_2, q_3)$.

Algebraic method: no drawing, only algebra on the 3 FK equations $x = p_x(q),\ y = p_y(q),\ z = p_z(q)$.
1. **Square and add** the equations so the angles cancel -> $q_3$
2. The $z$ equation has only $q_2$ -> $s_2$, then $c_2 = \pm\sqrt{1 - s_2^2}$ -> $q_2$
3. With $q_2, q_3$ known, the $x, y$ equations are **linear in $c_1, s_1$** -> solve the 2x2 system -> $q_1$

Compare with `ik_geometric.ipynb`: same answer, solved from a drawing instead of the equations.

In [16]:
import numpy as np
import sympy as sp
from IPython.display import display, Math
from roboticstoolbox import DHRobot, RevoluteMDH, PrismaticMDH
from spatialmath import SE3

def Rx(t): return sp.Matrix([[1, 0, 0, 0], [0, sp.cos(t), -sp.sin(t), 0], [0, sp.sin(t), sp.cos(t), 0], [0, 0, 0, 1]])
def Rz(t): return sp.Matrix([[sp.cos(t), -sp.sin(t), 0, 0], [sp.sin(t), sp.cos(t), 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
def Tx(d): return sp.Matrix([[1, 0, 0, d], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
def Tz(d): return sp.Matrix([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, d], [0, 0, 0, 1]])

# Modified DH (Craig): T^{i-1}_i = Rx(alpha_{i-1}) Tx(a_{i-1}) Rz(theta_i) Tz(d_i)
def mdh(a, alpha, d, theta):
    return Rx(alpha) * Tx(a) * Rz(theta) * Tz(d)

# display only: sin(q1) -> s_1, cos(q1 + q2) -> c_12
def short(e):
    def sub(f):
        terms = sp.Add.make_args(f.args[0])
        if not all(t.is_Symbol and t.name[0] == 'q' for t in terms):
            return f
        idx = ''.join(sorted((t.name[1:] for t in terms), key=int))
        return sp.Symbol(('s' if f.func == sp.sin else 'c') + '_{' + idx + '}')
    return e.replace(lambda t: t.func in (sp.sin, sp.cos), sub)

def show(lhs, rhs):
    tex = sp.latex(short(rhs)).replace(r'\operatorname{atan}_{2}', r'\operatorname{atan2}')
    display(Math(lhs + ' = ' + tex))

### 0 - Forward kinematics (position only)
Same MDH table as `fk.ipynb`.

In [17]:
q1, q2, q3 = sp.symbols('q1:4', real=True)
l1, l2, l3, l4 = sp.symbols('l1:5', positive=True)

# MDH table -> (a_{i-1}, alpha_{i-1}, d_i, theta_i)
mdh_table = [
    (0, 0,        l1,      q1),
    (0, sp.pi/2, -l2,      q2 + sp.pi/2),
    (0, sp.pi/2,  q3 + l3, 0),
]
T_tool = Tz(l4)
x, y, z = sp.symbols('x y z', real=True)   # target tip position

T0e = sp.eye(4)
for row in mdh_table:
    T0e = T0e * mdh(*row)
T0e = sp.simplify(T0e * T_tool)
p = T0e[0:3, 3]

show(r'p_e', p)

<IPython.core.display.Math object>

### 1 - The 3 FK equations
Joint 3 adds to $l_3$ and $l_4$, so write $L = l_3 + q_3 + l_4$. Unknowns are now $q_1, q_2, L$.

In [18]:
L = sp.Symbol('L', positive=True)
p_L = sp.simplify(p.subs(q3, L - l3 - l4))

for lhs_i, rhs_i in zip((x, y, z), p_L):
    show(sp.latex(lhs_i), rhs_i)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 2 - Square and add -> $q_3$
Move $l_1$ to the left in the $z$ equation, then square all 3 and add.
$c_1^2 + s_1^2 = 1$ and $c_2^2 + s_2^2 = 1$ make every angle disappear, leaving only $L$.
$L > 0$, so take the positive root, then $q_3 = L - l_3 - l_4$.

In [19]:
sum_sq = sp.simplify(sp.expand(p_L[0]**2 + p_L[1]**2 + (p_L[2] - l1)**2))
show(sp.latex(x**2 + y**2 + (z - l1)**2), sum_sq)

L_sol = sp.sqrt(x**2 + y**2 + (z - l1)**2 - l2**2)
q3_sol = L_sol - l3 - l4
show(r'L', L_sol)
show(r'q_3 = L - l_3 - l_4', q3_sol)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 3 - $z$ equation -> $q_2$
$z - l_1 = L s_2$ gives $s_2$. Then $c_2 = \pm\sqrt{1 - s_2^2}$, and
$1 - s_2^2 = \dfrac{L^2 - (z - l_1)^2}{L^2} = \dfrac{x^2 + y^2 - l_2^2}{L^2}$ (using step 2).

The $\pm$ gives **2 branches**. Use $q_2 = \operatorname{atan2}(s_2, c_2)$ (not $\arcsin$) so both branches come out right.
Reachable only if $x^2 + y^2 \ge l_2^2$.

In [20]:
s2_sol = sp.solve(sp.Eq(z, p_L[2]), sp.sin(q2))[0]
show(r's_2', s2_sol)
show(r'L^2 (1 - s_2^2) = L^2 - (z - l_1)^2', sp.expand(L_sol**2 - (z - l1)**2))

c2_sols = [sign * sp.sqrt(x**2 + y**2 - l2**2) / L for sign in (1, -1)]
for k, c2s in enumerate(c2_sols, start=1):
    show(rf'c_2^{{({k})}}', c2s)

# atan2(s2, c2): multiply both by L > 0, it cancels
q2_sols = [sp.atan2(z - l1, c2s * L) for c2s in c2_sols]
for k, s in enumerate(q2_sols, start=1):
    show(rf'q_2^{{({k})}}', s)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 4 - $x, y$ equations -> $q_1$
Treat $c_1, s_1$ as two unknowns and $c_2, L$ as known. The equations are linear:

$\begin{bmatrix} L c_2 & -l_2 \\ l_2 & L c_2 \end{bmatrix} \begin{bmatrix} c_1 \\ s_1 \end{bmatrix} = \begin{bmatrix} x \\ y \end{bmatrix}$

Solve the 2x2 system. Both answers share the denominator $L^2 c_2^2 + l_2^2 > 0$, which cancels inside $\operatorname{atan2}$.

In [21]:
C1, S1, C2 = sp.symbols('c_1 s_1 c_2', real=True)
lin = {sp.cos(q1): C1, sp.sin(q1): S1, sp.cos(q2): C2}
eq_x = sp.Eq(x, sp.expand(p_L[0]).subs(lin))
eq_y = sp.Eq(y, sp.expand(p_L[1]).subs(lin))
display(Math(sp.latex(eq_x)))
display(Math(sp.latex(eq_y)))

cs = sp.solve([eq_x, eq_y], [C1, S1])
show(r'c_1', sp.factor(cs[C1]))
show(r's_1', sp.factor(cs[S1]))

det = L**2 * C2**2 + l2**2
q1_gen = sp.atan2(sp.expand(sp.simplify(cs[S1] * det)), sp.expand(sp.simplify(cs[C1] * det)))
show(r'q_1', q1_gen)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 5 - IK solution in terms of $(x, y, z)$
Put $c_2$ of each branch into $q_1$ ($L$ cancels because only $L c_2$ appears).

In [22]:
solutions = []
for k, (c2s, q2s) in enumerate(zip(c2_sols, q2_sols), start=1):
    q1s = sp.atan2(*[sp.expand(a.subs(C2, c2s)) for a in q1_gen.args])
    sol = sp.Matrix([q1s, q2s, q3_sol])
    solutions.append(sol)
    display(Math(rf'\text{{Branch {k}}}:'))
    show(r'\begin{bmatrix} q_1 \\ q_2 \\ q_3 \end{bmatrix}', sol)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 5 - Check with numbers
Pick joint values, get the tip with FK, solve IK for that tip, then FK the IK answer again: the error should be ~0.

In [23]:
# edit here: link lengths (m)
vals = {l1: 0.64, l2: 0.2, l3: 0.22, l4: 0.095}

robot_mdh = DHRobot([
    RevoluteMDH(a=0.0, alpha=0.0,     d=vals[l1]),
    RevoluteMDH(a=0.0, alpha=np.pi/2, d=-vals[l2], offset=np.pi/2),
    PrismaticMDH(a=0.0, alpha=np.pi/2, offset=vals[l3]),
], name='my_mdh')
robot_mdh.tool = SE3.Tz(vals[l4])

ik = [sp.lambdify((x, y, z), sol.subs(vals), 'numpy') for sol in solutions]
wrap = lambda a: (a + np.pi) % (2 * np.pi) - np.pi   # angle -> [-pi, pi)

# edit here: joint values to test
for q_true in ([np.pi/6, np.pi/6, 0.18], [np.deg2rad(-25), np.deg2rad(-25), 0.135]):
    P = robot_mdh.fkine(q_true).t
    print('q =', np.round(q_true, 4), '-> tip P =', np.round(P, 4))
    for k, f in enumerate(ik, start=1):
        q_ik = np.array(f(*P), dtype=float).flatten()
        q_ik[:2] = wrap(q_ik[:2])
        err = np.linalg.norm(robot_mdh.fkine(q_ik).t - P)
        print(f'  branch {k}: q = {np.round(q_ik, 4)} | FK error = {err:.1e} m')

q = [0.5236 0.5236 0.18  ] -> tip P = [0.2713 0.3875 0.8875]
  branch 1: q = [0.5236 0.5236 0.18  ] | FK error = 5.6e-17 m
  branch 2: q = [-1.7449  2.618   0.18  ] | FK error = 1.4e-16 m
q = [-0.4363 -0.4363  0.135 ] -> tip P = [0.4542 0.0089 0.4498]
  branch 1: q = [-0.4363 -0.4363  0.135 ] | FK error = 3.5e-17 m
  branch 2: q = [-2.6661 -2.7053  0.135 ] | FK error = 1.4e-16 m


### Numeric example
Put a target $(x, y, z)$ through every step above (both branches), then FK the answer to check.

In [24]:
# helpers: plug numbers into a symbolic result and print it rounded
def val(e, subs):
    return float(sp.sympify(e).subs(subs).evalf())
def mat(e, subs, d=4):
    return sp.Matrix(e).subs(subs).evalf().applyfunc(lambda v: sp.Float(round(float(v), d)) if abs(float(v)) > 1e-12 else 0)
def showv(lhs, v, unit=''):
    display(Math(rf'{lhs} = {v:.4f}\ \text{{{unit}}}'))
def showa(lhs, v):
    display(Math(rf'{lhs} = {v:.4f}\ \text{{rad}} = {np.rad2deg(v):.2f}^\circ'))
def showm(lhs, M):
    display(Math(lhs + ' = ' + sp.latex(M)))

# edit here: target tip position (m). Here it comes from FK of a known pose, or type numbers directly
target = robot_mdh.fkine([np.pi/6, np.pi/6, 0.18]).t        # e.g. target = [0.27, 0.39, 0.89]

nums = {**vals, x: target[0], y: target[1], z: target[2]}
showm(r'\begin{bmatrix} x \\ y \\ z \end{bmatrix}', mat(sp.Matrix(target), {}))

# step 2: square and add
Lv = val(L_sol, nums)
showv(r'L = \sqrt{x^2 + y^2 + (z - l_1)^2 - l_2^2}', Lv, 'm')
showv(r'q_3 = L - l_3 - l_4', Lv - vals[l3] - vals[l4], 'm')

# step 3: z equation
nums[L] = Lv
s2v = val(s2_sol, nums)
showv(r's_2 = (z - l_1) / L', s2v)

for k, c2s in enumerate(c2_sols, start=1):
    display(Math(rf'\textbf{{Branch {k}}}:'))
    c2v = val(c2s, nums)
    showv(rf'c_2 = {"+" if k == 1 else "-"}\sqrt{{1 - s_2^2}}', c2v)
    q2v = np.arctan2(s2v, c2v)
    showa(r'q_2 = \operatorname{atan2}(s_2, c_2)', q2v)

    # step 4: x, y linear in c1, s1
    n_k = {**nums, C2: c2v}
    c1v, s1v = val(cs[C1], n_k), val(cs[S1], n_k)
    showv(r'c_1', c1v)
    showv(r's_1', s1v)
    q1v = np.arctan2(s1v, c1v)
    showa(r'q_1 = \operatorname{atan2}(s_1, c_1)', q1v)

    q3v = Lv - vals[l3] - vals[l4]
    err = np.linalg.norm(robot_mdh.fkine([q1v, q2v, q3v]).t - target)
    print(f'  FK check: |FK(q) - target| = {err:.1e} m')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

... (8 outputs left out)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

  FK check: |FK(q) - target| = 1.7e-16 m


### Same with Robotics Toolbox (numerical IK)
`ikine_LM` searches numerically instead of using a formula. Only the position can be chosen (3 joints), so `mask` keeps $x, y, z$ only.
The start guess `q0` decides which branch it finds.
The cell declares its own MDH robot first, so it can be copied and run on its own (lengths in m: l1 = 0.64, l2 = 0.2, l3 = 0.22, l4 = 0.095).


In [25]:
from roboticstoolbox import DHRobot, RevoluteMDH, PrismaticMDH
from spatialmath import SE3

# 1) declare the robot with MDH (a = a_{i-1}, alpha = alpha_{i-1}, d = d_i, offset added to the joint variable)
robot = DHRobot([
    RevoluteMDH(a=0.0, alpha=0.0,     d=0.64,                    qlim=[-np.pi, np.pi]),   # joint 1 (R)
    RevoluteMDH(a=0.0, alpha=np.pi/2, d=-0.2,  offset=np.pi/2,   qlim=[-np.pi, np.pi]),   # joint 2 (R)
    PrismaticMDH(a=0.0, alpha=np.pi/2, offset=0.22,              qlim=[0.0, 0.3]),        # joint 3 (P), m
], name='my_mdh')
robot.tool = SE3.Tz(0.095)                  # l4: last joint -> tip
print(robot)

# 2) use it
Tep = SE3.Trans(target)                     # wanted tip pose (only the position is used)
mask = [1, 1, 1, 0, 0, 0]                   # match x, y, z; ignore orientation

# edit here: start guesses (one near each branch)
for q0 in ([0.0, 0.0, 0.1], [-1.5, 2.5, 0.1]):
    sol = robot.ikine_LM(Tep, q0=q0, mask=mask, tol=1e-12)   # smaller tol = more exact
    q_sol = sol.q
    err = np.linalg.norm(robot.fkine(q_sol).t - target)
    print(f'q0 = {q0} -> success = {sol.success}')
    print(f'  q = [{np.rad2deg(q_sol[0]):.2f} deg, {np.rad2deg(q_sol[1]):.2f} deg, {q_sol[2]:.4f} m] | FK error = {err:.1e} m')

DHRobot: my_mdh, 3 joints (RRP), modified DH parameters
┌──────┬───────┬───────────┬────────────┬─────────┬────────┐
│ aⱼ₋₁ │ ⍺ⱼ₋₁  │    θⱼ     │     dⱼ     │   q⁻    │   q⁺   │
├──────┼───────┼───────────┼────────────┼─────────┼────────┤
│  0.0 │  0.0° │        q1 │       0.64 │ -180.0° │ 180.0° │
│  0.0 │ 90.0° │  q2 + 90° │       -0.2 │ -180.0° │ 180.0° │
│  0.0 │ 90.0° │      0.0° │  q3 + 0.22 │       0 │    0.3 │
└──────┴───────┴───────────┴────────────┴─────────┴────────┘

┌──────┬───────────────────────────────────────┐
│ tool │ t = 0, 0, 0.095; rpy/xyz = 0°, 0°, 0° │
└──────┴───────────────────────────────────────┘

q0 = [0.0, 0.0, 0.1] -> success = True
  q = [30.00 deg, 30.00 deg, 0.1800 m] | FK error = 4.8e-07 m
q0 = [-1.5, 2.5, 0.1] -> success = True
  q = [-99.98 deg, 150.00 deg, 0.1800 m] | FK error = 3.0e-07 m
